In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo 

covertype = fetch_ucirepo(id=31) # fetch dataset

x = covertype.data.features #features
y = covertype.data.targets  #class

In [4]:
x.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Wilderness_Area2,Wilderness_Area3,Wilderness_Area4
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,0
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,0
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,0
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,0
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,0


In [5]:
y.head()

,Cover_Type
0,5
1,5
2,2
3,2
4,5


In [6]:
print(x.shape,y.shape)

(581012, 54) (581012, 1)


In [7]:
y_new = y.iloc[:,0].to_numpy()

In [8]:
y_new.shape

(581012,)

In [9]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier

import warnings as wr
wr.filterwarnings('ignore')

In [10]:
# train test split
xtrain, xtest, ytrain, ytest = train_test_split(x, y_new, random_state=42, test_size=.30)

In [11]:
print(xtrain.shape,ytrain.shape)
print(xtest.shape,ytest.shape)

(406708, 54) (406708,)
(174304, 54) (174304,)


In [12]:
# test and validation
x_test, x_val, y_test, y_val = train_test_split(xtest, ytest, random_state=42, test_size=.50)

In [13]:
print(x_test.shape,y_test.shape)
print(x_val.shape,y_val.shape)

(87152, 54) (87152,)
(87152, 54) (87152,)


In [14]:
# random forest model
rf_model = RandomForestClassifier(random_state=42,n_jobs=-1,verbose=1)

In [15]:
rf_model.fit(xtrain,ytrain)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   50.5s finished


RandomForestClassifier(n_jobs=-1, random_state=42, verbose=1)

In [16]:
rf_y_predict = rf_model.predict(x_test)
rf_y_predict

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.4s finished


array([2, 2, 1, ..., 6, 2, 1])

In [17]:
rf_model.score(x_test,y_test)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    1.6s finished


0.9515559023315586

In [18]:
# classification report
print("=========>Random Forest Model Performance<=============") 
print("Accuracy:", accuracy_score(y_test, rf_y_predict),) 
print("----------Classification Report-------------") 
print(classification_report(y_test, rf_y_predict))

=========>Random Forest Model Performance<=============
Accuracy: 0.9515559023315586
----------Classification Report-------------
              precision    recall  f1-score   support

           1       0.96      0.94      0.95     31897
           2       0.95      0.97      0.96     42414
           3       0.94      0.96      0.95      5341
           4       0.94      0.86      0.90       401
           5       0.94      0.75      0.83      1448
           6       0.94      0.89      0.92      2636
           7       0.97      0.96      0.96      3015

    accuracy                           0.95     87152
   macro avg       0.95      0.90      0.92     87152
weighted avg       0.95      0.95      0.95     87152



In [19]:
# pre-prunning for cost_complexity
rf_params = {
    'n_estimators': [115, 130, 150, 175],
    'criterion': ['gini', 'entropy'], 
    'max_depth': [None, 15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
# Combinations: 4 × 2 × 4 × 3 × 3 × 2 = 576

In [20]:
# perform randomized search for random forest
rf_randomizedCV = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42),
                                    param_distributions=rf_params,n_iter=80,random_state=42,cv=3,
                                    n_jobs=-1,verbose=3)

In [21]:
rf_randomizedCV.fit(xtrain,ytrain)

Fitting 3 folds for each of 80 candidates, totalling 240 fits
[CV 2/3] END criterion=gini, max_depth=25, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=150;, score=0.912 total time= 2.9min
[CV 1/3] END criterion=entropy, max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=150;, score=0.924 total time= 2.7min
[CV 1/3] END criterion=gini, max_depth=25, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=150;, score=0.911 total time= 2.9min
[CV 2/3] END criterion=entropy, max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=10, n_estimators=150;, score=0.925 total time= 2.8min
[CV 1/3] END criterion=gini, max_depth=15, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=150;, score=0.808 total time= 1.8min
[CV 2/3] END criterion=gini, max_depth=15, max_features=log2, min_samples_leaf=1, min_samples_split=10, n_estimators=150;, score=0.811 total time= 1.8min
[CV 3/

RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42),
                   n_iter=80, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [None, 15, 20, 25],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [115, 130, 150, 175]},
                   random_state=42, verbose=3)

In [22]:
rf_cv_res = pd.DataFrame(rf_randomizedCV.cv_results_)

In [23]:
rf_cv_res

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_features,param_max_depth,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,163.934618,0.854537,11.979819,0.100569,150,5,2,sqrt,25,gini,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.911271,0.912074,0.913601,0.912315,0.000966,26
1,99.924814,0.790891,7.317159,0.791060,150,10,1,log2,15,gini,"{'n_estimators': 150, 'min_samples_split': 10,...",0.807959,0.810974,0.813541,0.810825,0.002281,80
2,154.069140,0.939931,11.010690,0.252538,150,10,2,log2,None,entropy,"{'n_estimators': 150, 'min_samples_split': 10,...",0.924371,0.924791,0.925720,0.924961,0.000564,15
3,126.087135,0.920682,8.248753,0.042424,150,5,4,log2,20,entropy,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.875452,0.879500,0.879323,0.878092,0.001868,48
4,105.245742,1.453288,6.945620,0.059300,150,5,1,log2,15,entropy,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.830132,0.830315,0.832233,0.830893,0.000950,61
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,171.248396,2.265348,9.601146,0.305244,130,10,2,sqrt,None,entropy,"{'n_estimators': 130, 'min_samples_split': 10,...",0.932935,0.932647,0.934771,0.933451,0.000941,6
76,176.371479,3.594910,10.489996,0.202850,130,2,1,sqrt,None,entropy,"{'n_estimators': 130, 'min_samples_split': 2, ...",0.945858,0.945777,0.946116,0.945917,0.000145,2
77,149.602585,1.659902,9.793317,0.134115,175,5,4,log2,20,gini,"{'n_estimators': 175, 'min_samples_split': 5, ...",0.858774,0.857497,0.863007,0.859759,0.002355,57
78,126.955864,2.955143,8.910018,0.051757,130,10,1,log2,25,gini,"{'n_estimators': 130, 'min_samples_split': 10,...",0.899078,0.901947,0.902942,0.901322,0.001638,41


[CV 2/3] END criterion=gini, max_depth=15, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=150;, score=0.823 total time= 1.9min
[CV 1/3] END criterion=gini, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=175;, score=0.933 total time= 4.0min
[CV 3/3] END criterion=gini, max_depth=15, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=115;, score=0.822 total time= 1.8min
[CV 1/3] END criterion=entropy, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=10, n_estimators=130;, score=0.933 total time= 3.0min
[CV 2/3] END criterion=entropy, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=130;, score=0.946 total time= 3.1min
[CV 3/3] END criterion=gini, max_depth=20, max_features=log2, min_samples_leaf=4, min_samples_split=5, n_estimators=175;, score=0.863 total time= 2.7min
[CV 1/3] END criterion=gini, max_depth=15, max_features=sqrt, min_sa

In [24]:
rf_cv_res.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_max_features,param_max_depth,param_criterion,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
50,221.524599,5.056351,14.441097,0.333088,175,2,1,sqrt,None,entropy,"{'n_estimators': 175, 'min_samples_split': 2, ...",0.945799,0.945777,0.946271,0.945949,0.000228,1
76,176.371479,3.594910,10.489996,0.202850,130,2,1,sqrt,None,entropy,"{'n_estimators': 130, 'min_samples_split': 2, ...",0.945858,0.945777,0.946116,0.945917,0.000145,2
63,189.134814,2.432092,12.629447,0.170996,150,2,1,sqrt,None,gini,"{'n_estimators': 150, 'min_samples_split': 2, ...",0.942605,0.942870,0.943837,0.943104,0.000529,3
38,188.107914,2.560804,12.989394,1.054142,150,5,1,sqrt,None,gini,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.939544,0.939182,0.940827,0.939851,0.000706,4
60,192.208183,1.571900,15.572270,0.099134,175,2,1,log2,None,gini,"{'n_estimators': 175, 'min_samples_split': 2, ...",0.936918,0.937759,0.938194,0.937624,0.000529,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35,111.149321,1.844393,5.967835,0.010326,130,10,1,sqrt,15,gini,"{'n_estimators': 130, 'min_samples_split': 10,...",0.817437,0.821574,0.820365,0.819792,0.001737,76
21,119.561321,2.292046,6.791695,0.019739,150,5,2,sqrt,15,gini,"{'n_estimators': 150, 'min_samples_split': 5, ...",0.817076,0.820652,0.820719,0.819482,0.001702,77
56,98.467857,1.466412,5.329249,0.030002,115,10,4,sqrt,15,gini,"{'n_estimators': 115, 'min_samples_split': 10,...",0.815453,0.818587,0.818402,0.817481,0.001436,78
45,103.872811,1.578487,6.940932,0.058560,150,2,1,log2,15,gini,"{'n_estimators': 150, 'min_samples_split': 2, ...",0.813322,0.814014,0.814736,0.814024,0.000578,79


In [25]:
rf_randomizedCV.best_score_

0.9459489367057413

In [26]:
rf_randomizedCV.best_params_

{'n_estimators': 175,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'max_depth': None,
 'criterion': 'entropy'}

In [27]:
best_model = rf_randomizedCV.best_estimator_

In [28]:
best_model

RandomForestClassifier(criterion='entropy', n_estimators=175, random_state=42)

In [29]:
y_pre = best_model.predict(x_test)

In [30]:
# classification report
print("=========>Randomized_search Best Model Performance<=============") 
print("Accuracy:", accuracy_score(y_test, y_pre),) 
print("----------Classification Report-------------")
print(classification_report(y_test, y_pre))

=========>Randomized_search Best Model Performance<=============
Accuracy: 0.9544015054158252
----------Classification Report-------------
              precision    recall  f1-score   support

           1       0.96      0.94      0.95     31897
           2       0.95      0.97      0.96     42414
           3       0.94      0.96      0.95      5341
           4       0.94      0.88      0.91       401
           5       0.94      0.78      0.85      1448
           6       0.94      0.90      0.92      2636
           7       0.97      0.95      0.96      3015

    accuracy                           0.95     87152
   macro avg       0.95      0.91      0.93     87152
weighted avg       0.95      0.95      0.95     87152

